# Kolokvijum I — Glavna knjiga

## Tekst zadatka

> Napraviti konstruktor tipa **GlavnaKnjiga** koji u sebi sadrži podatke naziv preduzeća, matični broj, pib i transakcije.
> Atribut transakcije je niz **Transakcija** objekata koji u sebi sadrže redni broj, broj računa, opis transakcije, status transakcije, datum, tip i iznos.
> **Redni broj transakcije je privatni statički atribut** čija vrednost se uvećava za jedan prilikom instanciranja svake transakcije.
> Status transakcije je string čije vrednosti mogu biti `"nerealizovana"`, `"realizovana"` i `"stornirana"`.
> Tip transakcije je string čije vrednosti mogu biti `"na teret"` i `"u korist"`.
>
> Glavna knjiga sadrži metode **dodajTransakciju** i **ukloniTransakciju**. Ove metode **ne smeju da mutiraju** originalni objekat nad kojim su primenjene.
> Rezultat ovih metoda je **nova instanca** glavne knjige u koju je dodata ili iz koje je uklonjena navedena transakcija.
> Novodobijena glavna knjiga sadrži i celokupnu **istoriju izmena** u vidu **reference na prethodnu glavnu knjigu** od koje je nastala.
> U glavnoj knjizi dodati **izvedeni atribut stanje** koji predstavlja razliku suma realizovanih transakcija u korist i na teret.
>
> Napraviti funkciju **pretraga** koja prima niz kriterijuma i glavnu knjigu, a kao rezultat vraća niz filtriran na osnovu kriterijuma.
> Kriterijumi su proizvoljno definisane predikatske funkcije. Primena kriterijuma pretrage se vrši onim redom kojim su navedeni u argumentima funkcije pretraga.
>
> Definisati funkciju **opoziv**, ova funkcija kao argumente prima glavnu knjigu i redni broj glavne knjige u istoriji.
> Rezultat funkcije je nova glavna knjiga koja u sebi sadrži sve transakcije koje je sadržala glavna knjiga na zadatom indeksu u istoriji,
> kao i sve transakcije koje su nastale nakon te glavne knjige, uz izmenu da su sve transakcije nastale nakon te glavne knjige statusa **stornirano**.
> Ova funkcija ne sme da menja nijednu od glavnih knjiga niti neku od njihovih transakcija.
>
> Zarad testiranja napraviti nekoliko instanci tipa GlavnaKnjiga, pozvati metode i funkcije nad njima i ispisati rezultate.

## Podela na logičke jedinice

| Jedinica | Šta sadrži | Glavni mehanizam |
| --- | --- | --- |
| 1 | `Transakcija`, dozvoljene vrednosti | IIFE + zatvorenje (privatni statički brojač), `Object.freeze` |
| 2 | čiste funkcije nad nizom transakcija | `reduce` u jednom prolazu, imutabilna kopija |
| 3 | `GlavnaKnjiga` i `istorija` | fabrička funkcija, `Object.defineProperty` + `get`, trajna (persistent) struktura |
| 4 | `pretraga` | funkcije višeg reda, predikati kao argumenti |
| 5 | `opoziv` | rad nad lancem verzija, `map` sa kopijom umesto izmene |
| 6 | test i ispis | provera imutabilnosti |

Odluke koje tekst zadatka ne propisuje, a moraju se doneti, označene su sa **Pretpostavka** u odgovarajućoj jedinici.

## Pravila funkcionalne paradigme kojih se rešenje drži

| Pravilo | Kako je sprovedeno u ovom rešenju |
| --- | --- |
| **nepromenljivost** | `Object.freeze` nad svakom transakcijom, nad nizom transakcija i nad knjigom; nigde `push`, `splice` ni dodela postojećem objektu |
| **bez `this` i bez `new`** | fabričke funkcije koje **vraćaju** objekat; sve metode su strelice nad zatvorenjem, pa ne mogu da izgube kontekst |
| **objekat se gradi jednim izrazom** | `Object.freeze(Object.defineProperties({ … }, { … }))` — nema dopisivanja polja posle stvaranja |
| **čiste funkcije** | `sa`, `bez`, `stanjeOd`, `stornirana`, `pretraga`, `opoziv` i `istorija` zavise samo od argumenata i ništa ne menjaju |
| **funkcije su vrednosti** | kriterijumi pretrage se prosleđuju kao argumenti, `preko(granica)` vraća funkciju |
| **preklapanje umesto petlje** | `reduce`, `map`, `filter` i rekurzija; u celom rešenju nema `for` petlje ni brojača |
| **bez deljenog promenljivog stanja** | svaka izmena daje novu verziju; stare verzije ostaju važeće i dele iste zamrznute transakcije |
| **efekti na ivici** | `console.log` postoji samo u jedinicama sa demonstracijom, nikad unutar funkcija koje računaju rezultat |

**Jedini izuzetak, i to zahtevan tekstom zadatka:** brojač rednog broja transakcije. „Privatni statički atribut koji se
uvećava pri svakom instanciranju" je po definiciji stanje, pa `Transakcija` **nije** referencijalno transparentna —
dva poziva sa istim argumentima daju različit redni broj. Stanje je zatvoreno u jednoj promenljivoj unutar IIFE-a
(jedinica 1) i nigde više se ne pojavljuje.

Provera dozvoljenih vrednosti (`status`, `tip`) izvedena je bacanjem izuzetka. To je jedini nelokalni izlaz u rešenju;
potpuno funkcionalna zamena bila bi povratna vrednost tipa „uspeh ili greška", što tekst zadatka ne traži.

---
# Jedinica 1 · `Transakcija` i privatni statički redni broj

In [ ]:
const STATUSI = Object.freeze(["nerealizovana", "realizovana", "stornirana"]);
const TIPOVI  = Object.freeze(["na teret", "u korist"]);

const Transakcija = (() => {
    let sledeciRedniBroj = 1;                       // privatni statički atribut — vidljiv samo unutar ovog zatvorenja

    return (brojRacuna, opis, status, datum, tip, iznos) => {
        if (!STATUSI.includes(status)) throw new Error(`Nedozvoljen status: ${status}`);
        if (!TIPOVI.includes(tip))     throw new Error(`Nedozvoljen tip: ${tip}`);

        return Object.freeze({
            redniBroj: sledeciRedniBroj++,          // uvećava se pri svakom instanciranju
            brojRacuna, opis, status, datum, tip, iznos
        });
    };
})();

const t1 = Transakcija("265-0001", "Uplata kupca",   "realizovana",   "2026-08-01", "u korist", 120000);
const t2 = Transakcija("265-0001", "Zakup poslovnog", "realizovana",  "2026-08-03", "na teret",  45000);

console.log("redni brojevi:", t1.redniBroj, t2.redniBroj);
console.log("transakcija:", t1);

try { Transakcija("265-0001", "Greska", "izmisljen", "2026-08-04", "u korist", 1); }
catch (g) { console.log("odbijeno:", g.message); }

try { t1.iznos = 999999; } catch (g) { console.log("izmena odbijena:", g.message); }
console.log("iznos posle pokušaja izmene:", t1.iznos, "| zamrznuta:", Object.isFrozen(t1));

console.log("brojač spolja:", typeof sledeciRedniBroj);   // ReferenceError bi bio da nije u typeof — nedostupan je

### Zašto ovaj mehanizam

**IIFE + zatvorenje.** „Privatni statički atribut" znači: jedna vrednost zajednička za sve transakcije, nedostupna spolja.
Odmah pozvana funkcija napravi promenljivu `sledeciRedniBroj` i vrati fabriku koja je jedina vidi — posle poziva IIFE-a
ne postoji nijedan izraz kojim bi se do te promenljive došlo. Brojač se ne prosleđuje kroz argumente niti stoji na objektu.

**`Object.freeze`.** Zadatak traži da `opoziv` ne sme da menja nijednu transakciju. Zamrznuta transakcija to garantuje
na nivou jezika, pa se ista transakcija može **deliti** između više verzija knjige bez odbrambenog kopiranja.

**Provera dozvoljenih vrednosti.** Status i tip su nabrojivi skupovi; provera pri konstrukciji znači da nijedna kasnija
funkcija ne mora da se brani od nepostojeće vrednosti.

### Funkcionalna paradigma prema OOP

| | Ovde (FP) | Kako bi bilo u OOP |
| --- | --- | --- |
| privatni statički | promenljiva u zatvorenju — nedostupna, bez sintakse za pristup | `static #brojac` u klasi; privatnost je novija i vezana za klasu |
| nepromenljivost | `Object.freeze`, deljenje bez kopiranja | setteri i „defensive copy" pri svakom prosleđivanju |
| tip | običan objekat, „patka" tipizacija | `instanceof Transakcija`, jasna identifikacija tipa |

**Prednosti.** Nema `this` i nema `new`, pa nema ni greške sa izgubljenim kontekstom. Nepromenljiv objekat se
bezbedno deli između verzija i niti. Konstrukcija i validacija su na jednom mestu.

**Mane.** Brojač je skriveno globalno stanje: fabrika **nije čista funkcija** — dva poziva sa istim argumentima daju
različit rezultat, i ne može se resetovati u testu (u OOP bi statičko polje bar bilo dostupno test kodu).
Gubi se `instanceof` provera i, sa njom, deo pomoći editora.

---
# Jedinica 2 · Čiste funkcije nad nizom transakcija

In [ ]:
const sa  = (niz, transakcija) => [...niz, transakcija];                     // dodavanje — nov niz
const bez = (niz, redniBroj)   => niz.filter(x => x.redniBroj !== redniBroj); // uklanjanje — nov niz

const stornirana = t => Object.freeze({ ...t, status: "stornirana" });        // kopija sa izmenjenim poljem

// razlika suma realizovanih: u korist minus na teret — u JEDNOM prolazu
const stanjeOd = niz => niz.reduce((s, x) =>
    x.status !== "realizovana" ? s :
    x.tip === "u korist"       ? s + x.iznos
                               : s - x.iznos, 0);

const probni = [t1, t2];

console.log("sa:  ", sa(probni, Transakcija("265-0002", "Kamata", "nerealizovana", "2026-08-05", "u korist", 500)).length,
            "| original:", probni.length);
console.log("bez: ", bez(probni, t1.redniBroj).map(x => x.opis), "| original:", probni.map(x => x.opis));
console.log("stanje:", stanjeOd(probni));
console.log("stornirana:", stornirana(t1).status, "| original:", t1.status);

### Zašto ovaj mehanizam

**Operacije su izdvojene iz tipa.** `sa`, `bez` i `stanjeOd` rade nad **nizom transakcija**, a ne nad glavnom knjigom.
Zbog toga ih koriste i metode knjige i funkcija `opoziv`, bez ijednog duplikata.

**Jedan `reduce` za stanje.** Definicija je „razlika dve sume", ali se ne pišu dva prolaza (`filter` pa `reduce`, dvaput):
predznak se bira u samom redjuseru, pa je posao završen jednim prolazom kroz niz.

**`stornirana` vraća kopiju.** `{ ...t, status: "stornirana" }` pravi nov objekat — original ostaje netaknut,
što je uslov iz zadatka za `opoziv`.

### Funkcionalna paradigma prema OOP

**Prednosti.** Ove funkcije su čiste: isti ulaz daje isti izlaz, nema skrivenog stanja, testiraju se bez ijedne instance
glavne knjige i bez lažnih objekata. Lako se kombinuju i ponovo koriste nad bilo kojim nizom transakcija.

**Mane.** Ponašanje je odvojeno od podataka: gledajući transakciju ne vidi se šta se sve nad njom može uraditi
(u OOP bi `storniraj()` bio metod tipa, pa je otkrivanje lakše). Ništa ne sprečava poziv `stanjeOd` nad nizom
koji uopšte nisu transakcije — nema enkapsulacije koja bi to zabranila.

---
# Jedinica 3 · `GlavnaKnjiga`, izvedeno stanje i istorija

**Pretpostavka.** „Redni broj glavne knjige u istoriji" tumači se kao **indeks u nizu verzija**, gde je `0` prva
(početna) knjiga, a poslednji indeks trenutna. Funkcija `istorija` upravo taj niz i vraća.

In [ ]:
const GlavnaKnjiga = (naziv, maticniBroj, pib, transakcije = [], prethodna = null) => {
    const t = Object.freeze([...transakcije]);                     // privatna, zamrznuta kopija

    const knjiga = Object.freeze(Object.defineProperties({          // ceo objekat nastaje JEDNIM izrazom
        naziv, maticniBroj, pib, prethodna,                        // javni podaci + veza sa prethodnom verzijom

        dodajTransakciju:  tr        => GlavnaKnjiga(naziv, maticniBroj, pib, sa(t, tr), knjiga),
        ukloniTransakciju: redniBroj => GlavnaKnjiga(naziv, maticniBroj, pib, bez(t, redniBroj), knjiga)
    }, {
        transakcije: { get: () => [...t],      enumerable: true },  // kopija ka spolja
        stanje:      { get: () => stanjeOd(t), enumerable: true }   // izvedeni atribut
    }));

    return knjiga;
};

// lanac verzija, od najstarije (indeks 0) do trenutne — rekurzija, bez petlje i bez menjanja niza
const istorija = knjiga => knjiga === null ? [] : [...istorija(knjiga.prethodna), knjiga];

const knjiga0 = GlavnaKnjiga("Merkur doo", "21456789", "108456789");
const knjiga1 = knjiga0.dodajTransakciju(t1);
const knjiga2 = knjiga1.dodajTransakciju(t2);

console.log("stanje po verzijama:", istorija(knjiga2).map(k => k.stanje));      // bez zagrada — izvedeni atribut
console.log("dužina istorije:", istorija(knjiga2).length);
console.log("knjiga0 netaknuta:", knjiga0.transakcije.length, "| knjiga2:", knjiga2.transakcije.length);
console.log("različite instance:", knjiga1 !== knjiga2, "| deljena transakcija:", knjiga1.transakcije[0] === knjiga2.transakcije[0]);

const knjiga3 = knjiga2.ukloniTransakciju(t2.redniBroj);
console.log("posle uklanjanja:", knjiga3.transakcije.map(x => x.opis), "| stanje:", knjiga3.stanje);
console.log("knjiga2 i dalje ima:", knjiga2.transakcije.map(x => x.opis));

knjiga2.transakcije.push(t1);                             // NAMERNO: dokaz da geter vraća kopiju, knjiga ostaje ista
console.log("posle push na kopiju:", knjiga2.transakcije.length);

### Zašto ovaj mehanizam

**Fabrička funkcija umesto `new`.** Objekat se pravi i vraća unutar funkcije, pa privatni niz `t` živi u zatvorenju.
Nema `this`, pa metode ne mogu da izgube kontekst kada se proslede dalje (`niz.map(k => k.stanje)` radi bez vezivanja).

**Objekat nastaje jednim izrazom.** `Object.freeze(Object.defineProperties({ … }, { … }))` daje gotovu, zamrznutu
knjigu odmah — nema koraka u kome objekat postoji nedovršen i prima polja dodelama. Metode se pozivaju na `knjiga`
iz zatvorenja, pa nova verzija dobija tačnu referencu na onu od koje je nastala.

**`istorija` je rekurzivna.** `knjiga === null ? [] : [...istorija(knjiga.prethodna), knjiga]` opisuje lanac verzija
kao definiciju, a ne kao postupak: nema brojača, nema `let`, nema niza koji se usput menja.

**`Object.defineProperty` sa `get`.** `stanje` je **izvedeni** atribut: ne čuva se, nego se računa iz transakcija pri
svakom čitanju, i čita se bez zagrada. Zbog toga ne može da se „razidje" sa sadržajem knjige — nema para vrednosti
koje treba držati usklađenim.

**Referenca `prethodna` = trajna struktura.** Svaka izmena pravi novu knjigu koja pokazuje na prethodnu.
Time je istorija dobijena samim načinom gradnje, bez posebnog dnevnika izmena. Nove verzije **dele** iste zamrznute
transakcije, pa kopiranje košta koliko i kopiranje niza pokazivača.

**`Object.freeze(o)` na kraju.** Sprečava da neko dopiše ili zameni metodu na gotovoj knjizi.

### Funkcionalna paradigma prema OOP

| | Ovde (FP) | Kako bi bilo u OOP |
| --- | --- | --- |
| izmena | nova instanca, stara ostaje | `this.transakcije.push(...)`, ista instanca |
| istorija | posledica gradnje (`prethodna`) | poseban `Memento` / `UndoStack` |
| identitet | svaka verzija je nov objekat | jedan objekat kroz ceo život, `===` stabilan |
| cena dodavanja | O(n) — kopira se niz | O(1) — `push` |

**Prednosti.** Nema mutacije, pa nema ni greške deljene reference: ko drži `knjiga1` siguran je da mu se sadržaj
neće promeniti pod rukama. Svaka verzija je konzistentan snimak, pogodan za poređenje, poništavanje i istovremeni
pristup iz više tokova. „Undo" i revizioni trag su besplatni.

**Mane.** Svaka izmena troši memoriju za nov objekat i nov niz, a lanac verzija raste neograničeno — u dugotrajnoj
aplikaciji mora se rezati. Identitet objekta se menja pri svakoj izmeni, pa je „ista knjiga" pojam koji mora da se
prati kroz promenljivu, a ne kroz referencu. Dodavanje jedne transakcije je O(n) umesto O(1).

---
# Jedinica 4 · `pretraga` — kriterijumi kao funkcije

In [ ]:
const pretraga = (kriterijumi, knjiga) =>
    kriterijumi.reduce((niz, kriterijum) => niz.filter(kriterijum), knjiga.transakcije);

// kriterijumi su obične predikatske funkcije — pišu se izvan pretrage
const realizovane = t => t.status === "realizovana";
const uKorist     = t => t.tip === "u korist";
const naTeret     = t => t.tip === "na teret";
const preko       = granica => t => t.iznos > granica;          // parametrizovan kriterijum
const uMesecu     = mesec   => t => t.datum.slice(0, 7) === mesec;

const knjigaP = ["Avans kupca|realizovana|u korist|2026-08-04|60000",
                 "Struja|realizovana|na teret|2026-08-06|18000",
                 "Nabavka robe|nerealizovana|na teret|2026-09-01|90000"]
    .map(red => red.split("|"))
    .reduce((k, [opis, status, tip, datum, iznos]) =>
        k.dodajTransakciju(Transakcija("265-0001", opis, status, datum, tip, Number(iznos))), knjiga2);

console.log("sve:", knjigaP.transakcije.length);
console.log("realizovane u korist:", pretraga([realizovane, uKorist], knjigaP).map(x => x.opis));
console.log("na teret preko 20000:", pretraga([naTeret, preko(20000)], knjigaP).map(x => x.opis));
console.log("avgust, realizovane:", pretraga([uMesecu("2026-08"), realizovane], knjigaP).map(x => x.opis));
console.log("bez kriterijuma:", pretraga([], knjigaP).length);

// kriterijumi stižu kao LISTA, pa se lista može sastaviti i u toku rada — npr. iz popunjenih polja pretrage
const popunjeno = { status: "realizovana", tip: "na teret", minIznos: 10000 };
const izPolja = [
    ...(popunjeno.status   ? [t => t.status === popunjeno.status] : []),
    ...(popunjeno.tip      ? [t => t.tip === popunjeno.tip]       : []),
    ...(popunjeno.minIznos ? [preko(popunjeno.minIznos)]          : [])
];
console.log("lista sastavljena u toku rada:", pretraga(izPolja, knjigaP).map(x => x.opis));

### Zašto ovaj mehanizam

**Predikat kao argument.** Kriterijum je funkcija koja vraća `true`/`false`, pa `filter` ne mora ništa da zna o
transakcijama. Nov kriterijum se dodaje pisanjem nove funkcije — postojeći kod se ne dira.

**`reduce` preko liste kriterijuma.** Zadatak traži primenu **onim redom kojim su navedeni**. `reduce` kreće od
niza svih transakcija u prosleđenoj knjizi i svaki sledeći kriterijum primenjuje na rezultat prethodnog, čime je
redosled doslovno sadržan u redosledu koraka. Prazan niz kriterijuma prirodno daje sve transakcije, bez posebne grane.

**Kriterijumi su lista, ne pojedinačni argumenti.** Zato pozivalac može da je **sastavi u toku rada** — iz popunjenih
polja pretrage, iz podešavanja, iz konfiguracije — i prosledi kao vrednost. Sa `...kriterijumi` (rest) to bi tražilo
rasipanje na mestu poziva, a prazna pretraga ne bi imala prirodan oblik.

**`preko(granica)`** je funkcija koja vraća funkciju — parametar se zaključa u zatvorenju, pa kriterijum i dalje
ima potpis koji `filter` očekuje.

### Funkcionalna paradigma prema OOP

**Prednosti.** Nema ni jedne klase po kriterijumu i nema grananja po „tipu pretrage": kombinacije se prave na mestu
poziva. U OOP se isti efekat postiže obrascem *Specification* ili *Strategy* — po klasu za svaki uslov, plus
kompozitne klase za `and`/`or`.

**Mane.** Svaki kriterijum pravi **nov međuniz**, pa je pretraga sa `n` kriterijuma `n` prolaza kroz podatke
(u direktorijumu `k1kaok2` isti posao radi kompozicija transdjusera u jednom prolazu). Kada rezultat bude prazan,
iz koda se ne vidi **koji** kriterijum ga je ispraznio — nema imena ni poruke, jer je kriterijum anonimna funkcija.

---
# Jedinica 5 · `opoziv` — nova knjiga iz zadate verzije

**Pretpostavka.** „Transakcije nastale nakon te glavne knjige" su one koje se pojavljuju u verzijama posle zadatog
indeksa, a nisu postojale u toj verziji — prepoznaju se po **rednom broju**. Nova knjiga nastaje od **trenutne**
verzije, pa je njena `prethodna` upravo prosleđena knjiga i istorija ostaje neprekinuta.

In [ ]:
const opoziv = (knjiga, redniBrojUIstoriji) => {
    const verzije = istorija(knjiga);
    const stara = verzije[redniBrojUIstoriji];
    if (!stara) throw new Error(`Ne postoji verzija sa indeksom ${redniBrojUIstoriji}`);

    const bileRanije = new Set(stara.transakcije.map(x => x.redniBroj));

    const nastaleKasnije = verzije
        .slice(redniBrojUIstoriji + 1)
        .flatMap(k => k.transakcije)                                            // sve iz kasnijih verzija
        .filter(x => !bileRanije.has(x.redniBroj))                              // samo nove
        .filter((x, i, niz) => niz.findIndex(y => y.redniBroj === x.redniBroj) === i)  // bez ponavljanja
        .map(stornirana);                                                       // kopije, originali netaknuti

    return GlavnaKnjiga(knjiga.naziv, knjiga.maticniBroj, knjiga.pib,
                        [...stara.transakcije, ...nastaleKasnije], knjiga);
};

const opozvana = opoziv(knjigaP, 2);          // vraćamo se na verziju sa indeksom 2

console.log("verzija 2 je imala:", istorija(knjigaP)[2].transakcije.map(x => x.opis));
console.log("opozvana sadrži:", opozvana.transakcije.map(x => `${x.opis}(${x.status})`));
console.log("stanje pre:", knjigaP.stanje, "| stanje posle opoziva:", opozvana.stanje);
console.log("istorija posle opoziva:", istorija(opozvana).length, "verzija");

console.log("originalna knjiga netaknuta:", knjigaP.transakcije.map(x => x.status));
console.log("originalne transakcije netaknute:", istorija(knjigaP)[3].transakcije.every(x => x.status !== "stornirana"));

try { opoziv(knjigaP, 99); } catch (g) { console.log("odbijeno:", g.message); }

### Zašto ovaj mehanizam

**Ništa se ne vraća unazad.** Pošto je svaka verzija nepromenljiva i još uvek postoji, „opoziv" ne mora da poništava
izmene — dovoljno je pročitati staru verziju i **sastaviti novu** od njenog sadržaja i storniranih kopija kasnijih
transakcija. Zbog toga u celoj funkciji nema nijedne dodele postojećem objektu.

**`Set` rednih brojeva.** Poređenje po identitetu objekta ne bi radilo za transakcije koje su prošle kroz kopiranje,
a redni broj je jedinstven po definiciji iz zadatka — zato je on ključ za „ova je već postojala".

**`map(stornirana)`.** Storniranje se izvodi kao preslikavanje u nove objekte. Original ostaje u svojoj verziji
netaknut, čime je ispunjen izričit zahtev da funkcija ne sme da menja nijednu knjigu ni transakciju.

**Dedupliciranje.** Ista transakcija se pojavljuje u svakoj verziji nastaloj posle njenog dodavanja, pa se kroz
`flatMap` prirodno javlja više puta; filter po prvom pojavljivanju zadržava po jedan primerak.

### Funkcionalna paradigma prema OOP

**Prednosti.** Poništavanje izmena ne traži poseban obrazac (*Memento*, *Command* sa `undo()`), niti snimanje stanja
pre izmene — istorija je već posledica načina na koji su knjige nastajale. Nemoguće je slučajno oštetiti staru verziju,
jer je sve zamrznuto.

**Mane.** Obilazak istorije je linearan i ide unazad kroz lanac, pa je pristup „verziji broj `k`" O(n) — u OOP bi
`ArrayList` snimaka dao O(1). Cela istorija se drži u memoriji dokle god postoji referenca na poslednju knjigu:
nijedna verzija ne može da se oslobodi, jer svaka drži prethodnu.

---
# Jedinica 6 · Test i ispis

In [ ]:
console.log("═══ 1. instanciranje ═══");
const mercur = ["Prodaja usluga|realizovana|u korist|2026-08-10|150000",
                "Plate|realizovana|na teret|2026-08-11|95000",
                "Reprezentacija|nerealizovana|na teret|2026-08-12|12000"]
    .map(red => red.split("|"))
    .reduce((k, [opis, status, tip, datum, iznos]) =>
        k.dodajTransakciju(Transakcija("170-9999", opis, status, datum, tip, Number(iznos))),
        GlavnaKnjiga("Vega ad", "20099887", "105566778"));

console.log(mercur.naziv, "| mb:", mercur.maticniBroj, "| pib:", mercur.pib);
console.log("transakcije:", mercur.transakcije.map(x => `${x.redniBroj}:${x.opis}`));

console.log("═══ 2. izvedeni atribut stanje ═══");
console.log("stanje:", mercur.stanje, "(150000 - 95000; nerealizovana se ne računa)");

console.log("═══ 3. imutabilnost metoda ═══");
const dodata = mercur.dodajTransakciju(Transakcija("170-9999", "Kamata", "realizovana", "2026-08-13", "u korist", 4000));
const uklonjena = mercur.ukloniTransakciju(mercur.transakcije[1].redniBroj);
console.log("posle dodavanja:", dodata.stanje, "| posle uklanjanja:", uklonjena.stanje, "| original:", mercur.stanje);
console.log("nove instance:", mercur !== dodata && mercur !== uklonjena);

console.log("═══ 4. istorija ═══");
console.log("verzija:", istorija(dodata).length, "| stanja:", istorija(dodata).map(k => k.stanje));

console.log("═══ 5. pretraga ═══");
console.log("realizovane na teret:", pretraga([realizovane, naTeret], dodata).map(x => x.opis));
console.log("preko 100000:", pretraga([preko(100000)], dodata).map(x => x.opis));

console.log("═══ 6. opoziv ═══");
const vracena = opoziv(dodata, 2);
console.log("opozvana:", vracena.transakcije.map(x => `${x.opis}(${x.status})`));
console.log("stanje:", vracena.stanje, "| original:", dodata.stanje);
console.log("ništa nije mutirano:", dodata.transakcije.every(x => x.status !== "stornirana"));

### Zašto ovaj mehanizam

Zadatak izričito traži ispis rezultata. Test ide redom kojim su zahtevi navedeni i za svaki proverava **posmatranu
posledicu**, ne implementaciju: da stanje ne računa nerealizovane, da metode vraćaju nove instance a original ostaje
isti, da istorija ima očekivanu dužinu i da opoziv ne menja zatečene podatke.

### Funkcionalna paradigma prema OOP

**Prednosti.** Nema pripreme okruženja: sve funkcije su čiste ili prave nove vrednosti, pa se test svodi na
poređenje ulaza i izlaza — bez lažnih objekata, bez `setUp`/`tearDown`, bez redosleda izvršavanja koji utiče na ishod.
Original i rezultat postoje **istovremeno**, pa se imutabilnost proverava jednim poređenjem.

**Mane.** Pošto ne postoji promenljivo stanje koje bi se ispitalo „iznutra", svaka provera mora da se izrazi kroz
povratne vrednosti; kod grešaka se dobija vrednost koja ne valja, ali ne i mesto u lancu gde je nastala.
Ispisivanje međukoraka traži dodatni kod (u `k2` to rešava transdjuser za ispis).